# 03 · Extensión experimental

**Condición modificada (una sola):** eliminar las variables más informativas de los datos que recibe el clasificador.

**Pregunta:** si las etiquetas de riesgo se crearon con K-Means sobre ciertas variables, ¿qué tanto depende el desempeño casi perfecto de tener esas mismas variables al clasificar?

**Diseño:**
- Las etiquetas `RIESGO_DE_MOROSIDAD` **no cambian** (las del notebook 02).
- Se eliminan las k variables más importantes, con k = 1, 3 y 5.
- Se repite exactamente el mismo procedimiento: 80/20 estratificado, SMOTE, los 4 modelos, las 9 métricas.

Ni el paper ni sus pruebas de estrés evalúan este escenario.

In [ ]:
import sys
sys.path.append("../src")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.precision", 4)
from sklearn.ensemble import RandomForestClassifier
from modelos import extension_sin_variables
from utilidades import cache_csv
import paper

RECALCULAR = False
SEMILLA = 42

In [ ]:
datos = pd.read_csv("../data/cartera_etiquetada.csv")
X, y = datos.drop(columns="RIESGO_DE_MOROSIDAD"), datos["RIESGO_DE_MOROSIDAD"]
base = pd.read_csv("../results/02_metricas_replica.csv", index_col=0)
print(X.shape, "| proporción de clases:", y.value_counts(normalize=True).sort_index().round(3).tolist())

## Ranking de variables para predecir el riesgo

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=SEMILLA, n_jobs=-1).fit(X, y)
ranking = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
ranking.head(8).round(3)

## Experimento

In [ ]:
extension = cache_csv("../results/03_extension.csv", extension_sin_variables, RECALCULAR,
                      X=X, y=y, ranking=list(ranking.index), seed=SEMILLA)
extension[["Variables_eliminadas", "Eliminadas"]].drop_duplicates().set_index("Variables_eliminadas")

In [ ]:
base_k0 = base.assign(Variables_eliminadas=0).rename_axis("Modelo").reset_index()
todo = pd.concat([base_k0, extension], ignore_index=True)
tabla = todo.pivot(index="Variables_eliminadas", columns="Modelo", values="F1 Macro")[paper.TABLA_2.index]
tabla.to_csv("../results/03_f1_por_variables_eliminadas.csv")
tabla

In [ ]:
resumen = todo[todo.Modelo == "GradientBoosting"].set_index("Variables_eliminadas")[
    ["Accuracy", "Balanced Accuracy", "F1 Macro", "Matthews Corrcoef", "AUC ROC (OVR)"]]
resumen

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
tabla.plot(ax=axes[0], marker="o")
axes[0].set(title="F1 Macro según variables eliminadas", xlabel="variables más informativas eliminadas (k)",
            ylabel="F1 Macro", xticks=tabla.index)
todo.pivot(index="Variables_eliminadas", columns="Modelo", values="Matthews Corrcoef")[paper.TABLA_2.index].plot(ax=axes[1], marker="o")
axes[1].set(title="MCC según variables eliminadas", xlabel="k", ylabel="MCC", xticks=tabla.index)
for ax in axes:
    ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.savefig("../figures/03_extension.png", dpi=120); plt.show()

## Interpretación

- Con k = 0 se obtiene la réplica del notebook 02 (métricas casi perfectas).
- A medida que se retiran las variables que también definieron los clústeres, el desempeño cae: el modelo pierde la información con la que se trazaron las fronteras de las clases.
- Esto confirma que las métricas casi perfectas del paper reflejan sobre todo la **consistencia entre el clustering y el clasificador**, no necesariamente capacidad de anticipar la mora.
- La caída no es igual para todos los modelos: compara si los ensembles (GB, Voting) siguen siendo más robustos que AdaBoost, como afirma el paper en sus pruebas de estrés.

**Implicación para el proyecto de grado (conciliación AP-AR):** cuando la etiqueta se construye a partir de los mismos datos que el modelo (por ejemplo, clasificar partidas "conciliables" con reglas sobre montos y fechas y luego predecirlas con esos mismos montos y fechas), una accuracy alta no garantiza utilidad. Conviene validar contra una etiqueta independiente, como conciliaciones reales hechas por el equipo.